# Athena Database and Table Creation

This notebook will establish the AWS Athena Database that will be responsible for developing the raw_data and processed_data tables which will store the data.

In [1]:
# import libraries
import boto3
import time

In [2]:
# run athena query function
def run_athena_query(query, database, output_location):
    athena_client = boto3.client('athena')
    # query execution.
    response = athena_client.start_query_execution(
        QueryString=query,
        QueryExecutionContext={'Database': database},
        ResultConfiguration={'OutputLocation': output_location}
    )
    query_execution_id = response['QueryExecutionId']
    print(f"Query submitted. Execution ID: {query_execution_id}")
    # poll query until it completes.
    state = 'RUNNING'
    while state in ['RUNNING', 'QUEUED']:
        response = athena_client.get_query_execution(QueryExecutionId=query_execution_id)
        state = response['QueryExecution']['Status']['State']
        print(f"Current query state: {state}")
        if state in ['RUNNING', 'QUEUED']:
            time.sleep(2)
    
    if state == 'FAILED':
        reason = response['QueryExecution']['Status'].get('StateChangeReason', 'Unknown error')
        raise Exception(f"Query {query_execution_id} failed: {reason}")
    
    print(f"Query {query_execution_id} completed successfully.")
    return query_execution_id

In [3]:
# set athena database name
athena_database = 'processed_data'
# set bucket used by Athena to store query results.
output_location = 's3://group9-ml-proj-athena-query-bucket-grw/'  # change your initials

# create db query
create_db_query = f"CREATE DATABASE IF NOT EXISTS {athena_database};"
print("Creating database...")
run_athena_query(create_db_query, database='default', output_location=output_location)

Creating database...
Query submitted. Execution ID: 1f46bdd2-bf96-4e2d-a0b7-8802aa7ee779
Current query state: QUEUED
Current query state: SUCCEEDED
Query 1f46bdd2-bf96-4e2d-a0b7-8802aa7ee779 completed successfully.


'1f46bdd2-bf96-4e2d-a0b7-8802aa7ee779'

In [4]:
# create external table
create_table_query = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {athena_database}.processed_data (
    FL_DATE STRING,
    CRS_DEP_TIME STRING,
    OP_UNIQUE_CARRIER STRING,
    DEST STRING,
    wind_dir_degrees DOUBLE,
    wind_speed_kt DOUBLE,
    wind_gust_kt DOUBLE,
    visibility_statute_mi DOUBLE,
    temperature_c DOUBLE,
    dewpoint_c DOUBLE,
    altimeter_hpa DOUBLE,
    Flight_Status STRING,
    Carrier_AA BOOLEAN,
    Carrier_AS BOOLEAN,
    Carrier_B6 BOOLEAN,
    Carrier_DL BOOLEAN,
    Carrier_F9 BOOLEAN,
    Carrier_G4 BOOLEAN,
    Carrier_HA BOOLEAN,
    Carrier_NK BOOLEAN,
    Carrier_OO BOOLEAN,
    Carrier_UA BOOLEAN,
    Carrier_WN BOOLEAN,
    DEST_REGION STRING,
    Dest_Region_Midwest BOOLEAN,
    Dest_Region_Mountain BOOLEAN,
    Dest_Region_Northeast BOOLEAN,
    Dest_Region_OCONUS BOOLEAN,
    Dest_Region_South BOOLEAN,
    Dest_Region_Southwest BOOLEAN,
    Dest_Region_West BOOLEAN,
    Flight_Status_Binary BOOLEAN    
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
  'separatorChar' = ',',
  'quoteChar' = '\"'
)
LOCATION 's3://group9-ml-proj-processed-data-bucket-grw/all_data/'
TBLPROPERTIES ('skip.header.line.count'='1');
"""
print("Creating external table...")
run_athena_query(create_table_query, database=athena_database, output_location=output_location)

print("Athena database and table setup completed successfully.")

Creating external table...
Query submitted. Execution ID: e4362e9e-e2bb-4220-a981-aa1d309964aa
Current query state: QUEUED
Current query state: SUCCEEDED
Query e4362e9e-e2bb-4220-a981-aa1d309964aa completed successfully.
Athena database and table setup completed successfully.


In [5]:
%%html
<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>